### Imports

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
from shapely.ops import unary_union
from shapely.geometry import Polygon

### --- 1. Load Data ---

In [ ]:
# Set up base path and crs
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")
jamaica_metric_grid_crs = "EPSG:3448"

# Load the Jamaica boundary (optional, for context)

jamaica_boundary_path = base_path / "Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)

print(jamaica_boundary.crs)

# Load land use data

land_use = base_path / "2013_landuse_LandCover.shp"
terrestrial_landcover = gpd.read_file(land_use)

# Reproject to Jamaica Metric Grid (EPSG:3448)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print("Landcover CRS:", terrestrial_landcover.crs)


### --- 2. Filter to Primary Broadleaved Forest Patches -------

### Subset to "Closed broadleaved forest (Primary Forest)"

In [ ]:
print(terrestrial_landcover.columns)

In [ ]:
primary_broadleaved_forest_patches = terrestrial_landcover[
    terrestrial_landcover["Classify"] == "Closed broadleaved forest (Primary Forest)"
].copy()

In [ ]:
display(primary_broadleaved_forest_patches.head())

### --- 3. Calculate Area & Rank by Size ----

In [ ]:
# Calculate the area of each patch
primary_broadleaved_forest_patches["area_m2"] = primary_broadleaved_forest_patches.geometry.area

# Rank patches by area (largest = rank 1)
primary_broadleaved_forest_patches["area_rank"] = primary_broadleaved_forest_patches["area_m2"].rank(method="dense", ascending=False)

ranked_patches = primary_broadleaved_forest_patches.sort_values(by="area_rank")

In [ ]:
# Define the columns you want to display
columns_to_display = ["OBJECTID", "Classify", "area_m2", "area_rank", "geometry"]

In [ ]:
# Display the ranked patches with the selected columns
display(ranked_patches[columns_to_display].head())

In [ ]:
primary_broadleaved_forest_patches.plot()

In [ ]:

# Discrete colors for the top 5 ranks
top_5_colors = {
    1: "red",   # Largest
    2: "orange",
    3: "yellow",
    4: "green",
    5: "blue"
}

# Fallback colormap for ranks > 5
fallback_cmap = cm.get_cmap("viridis", len(ranked_patches) - 5)
fallback_norm = mcolors.Normalize(vmin=6, vmax=ranked_patches["area_rank"].max())

# Assign colors: Top 5 get discrete colors, others use fallback colormap, and NaN get grey
def assign_color(rank):
    if np.isnan(rank):  # Check for NaN
        return "grey"   # Color for unranked patches
    elif rank in top_5_colors:
        return top_5_colors[rank]
    else:
        return fallback_cmap(fallback_norm(rank))

ranked_patches["color"] = ranked_patches["area_rank"].apply(assign_color)

# Plot as before
fig, ax = plt.subplots(figsize=(10, 8))
base = jamaica_boundary.plot(ax=ax, color="lightgrey", edgecolor="black")  # Plot boundary
ranked_patches.plot(ax=ax, color=ranked_patches["color"], edgecolor="black", alpha=0.8)

# Add a colorbar for fallback colormap
sm = cm.ScalarMappable(cmap=fallback_cmap, norm=fallback_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.04)
cbar.set_label('Forest Rank > 5', fontsize=12)

# Add a legend for the top 5 ranks and unranked patches
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Rank 1 (Largest)', markerfacecolor='red', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Rank 2', markerfacecolor='orange', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Rank 3', markerfacecolor='yellow', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Rank 4', markerfacecolor='green', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Rank 5', markerfacecolor='blue', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Unranked', markerfacecolor='grey', markersize=10)  # For unranked
]
ax.legend(handles=legend_elements, title="Forest Size Rank", loc="upper left")

# Add title and labels
ax.set_title("Ranked Closed Broadleaved Forest Patches", fontsize=16)
ax.set_xlabel("Easting (meters)")
ax.set_ylabel("Northing (meters)")

plt.show()

### --- 4. Fix Invalid Geometries (if any) -----

In [ ]:
# Buffer(0) is a common trick to fix self-intersections
primary_broadleaved_forest_patches["geometry"] = (
    primary_broadleaved_forest_patches.geometry.buffer(0)
)
print(
    "Valid geometries:",
    primary_broadleaved_forest_patches.is_valid.sum(),
    "/",
    len(primary_broadleaved_forest_patches),
)


### --- 5. Nearest Patch Calculation ----

In [ ]:
# We'll store both the nearest patch ID and the distance in meters.

In [ ]:
primary_broadleaved_forest_patches["nearest_patch_id"] = np.nan
primary_broadleaved_forest_patches["nearest_dist_m"] = np.nan

for i, row in primary_broadleaved_forest_patches.iterrows():
    # Drop the current patch from comparison
    others = primary_broadleaved_forest_patches.drop(i)
    
    # Calculate distances from this patch to all others
    dist_series = others.geometry.distance(row.geometry)
    
    # Identify the minimum distance and index
    min_dist = dist_series.min()            # The smallest distance
    min_idx = dist_series.idxmin()          # The index of that patch in 'others'
    
    # Store results in the main GeoDataFrame
    primary_broadleaved_forest_patches.at[i, "nearest_patch_id"] = min_idx
    primary_broadleaved_forest_patches.at[i, "nearest_dist_m"] = min_dist


In [ ]:
# --- 6. Quick Check of Results ---------------------------------------------

print(
    primary_broadleaved_forest_patches[
        ["area_rank", "nearest_patch_id", "nearest_dist_m"]
    ].head()
)


In [ ]:
# --- 7. Plotting -----------------------------------------------------------
# We'll plot:
#   A) The boundary of Jamaica (for context).
#   B) Our forest patches, colored by nearest_dist_m.
#   C) Lines from each patch to its nearest patch (centroid to centroid).

fig, ax = plt.subplots(figsize=(10, 8))

# A) Jamaica boundary (optional for context)
jamaica_boundary.plot(ax=ax, color="lightgrey", edgecolor="black")

# B) Plot forest patches, color by nearest_dist_m
primary_broadleaved_forest_patches.plot(
    column="nearest_dist_m",
    cmap="viridis",
    legend=True,
    edgecolor="black",
    alpha=0.8,
    ax=ax
)

# C) Draw lines from each patch to its nearest patch
for i, row in primary_broadleaved_forest_patches.iterrows():
    # Skip if no nearest patch found
    if pd.isna(row["nearest_patch_id"]):
        continue
    
    # Current patch centroid
    x1, y1 = row.geometry.centroid.x, row.geometry.centroid.y
    
    # Retrieve the nearest patch
    nearest_id = int(row["nearest_patch_id"])  # Convert float to int index
    nearest_row = primary_broadleaved_forest_patches.loc[nearest_id]
    
    # Nearest patch centroid
    x2, y2 = nearest_row.geometry.centroid.x, nearest_row.geometry.centroid.y
    
    # Draw a line between the two centroids
    ax.plot([x1, x2], [y1, y2], color="red", linewidth=1, alpha=0.7)

# Title & Show
ax.set_title("Nearest Patch Distances for Primary Broadleaved Forest", fontsize=14)
plt.show()

In [ ]:
# 1. Filter or Identify "Convertible" Polygons
#    Let's assume you have other land uses in 'terrestrial_landcover'.
#    Suppose these classes are convertible to forest:
convertible_classes = ["Fields: Bare Land", "Quarry", "Bauxite Extraction", "Bamboo", "Fields: Herbaceous crops, fallow, cultivated vegetables", "Fields: Pasture,Human disturbed, grassland", "Bamboo and Fields", "Fields  and Bamboo"]  # example list

convertible_polygons = terrestrial_landcover[
    terrestrial_landcover["Classify"].isin(convertible_classes)
].copy()

print("Convertible polygons:", len(convertible_polygons))


In [ ]:
# 2. Current Forest: get a unary union of all forest patches
current_forest_union = unary_union(primary_broadleaved_forest_patches.geometry)

def count_patches_in_union(union_geom):
    """
    Count how many distinct polygon-patches are in this unioned geometry.
    If it's MultiPolygon, each sub-polygon is a patch.
    """
    if union_geom.is_empty:
        return 0
    if union_geom.geom_type == "Polygon":
        return 1
    elif union_geom.geom_type == "MultiPolygon":
        return len(list(union_geom.geoms))
    else:
        # In unusual cases, might have geometrycollection
        # Handle those if needed
        return np.nan

current_patch_count = count_patches_in_union(current_forest_union)
print(f"Current forest patch count: {current_patch_count}")

In [ ]:
# 3. Scenario Testing:
#    For each convertible polygon, see how the patch count changes if we "plant" it.
improvements = []

for idx, row in convertible_polygons.iterrows():
    poly = row.geometry
    
    # Merge this convertible polygon into the existing forest union
    # 'union_geom' is the resulting geometry if that polygon were reforested
    scenario_union = current_forest_union.union(poly)
    
    # Count how many patches remain
    scenario_patch_count = count_patches_in_union(scenario_union)
    
    # Improvement is the difference in patch count
    # (how many fewer patches are there than before?)
    improvement = current_patch_count - scenario_patch_count
    
    improvements.append({
        "convertible_idx": idx,         # index in 'convertible_polygons'
        "land_use_class": row["Classify"],
        "patch_count_after": scenario_patch_count,
        "patch_count_improvement": improvement
    })

results_df = pd.DataFrame(improvements)
results_df.sort_values("patch_count_improvement", ascending=False, inplace=True)

print("Top candidates for reforestation:")
print(results_df.head(10))

In [ ]:
## Visualizing Where the “Best” Polygons Are

# Let's define an arbitrary threshold or just look at the top N
top_candidates = results_df[results_df["patch_count_improvement"] > 0]
top_ids = top_candidates["convertible_idx"].unique()

fig, ax = plt.subplots(figsize=(10, 8))

# Plot the existing forest patches in green
primary_broadleaved_forest_patches.plot(ax=ax, color="green", edgecolor="black", alpha=0.6)

# Plot the "top" convertible polygons in red
# We'll subset 'convertible_polygons' by those indices
convertible_polygons.loc[top_ids].plot(ax=ax, color="red", edgecolor="black", alpha=0.5)

ax.set_title("Potential Reforestation Polygons to Improve Forest Connectivity")
plt.show()

In [ ]:
# 3. Create a unary union of the existing forest
forest_union = unary_union(primary_broadleaved_forest_patches.geometry)

def count_patches_in_union(geom):
    """Return the number of distinct polygon patches in a union geometry."""
    if geom.is_empty:
        return 0
    if geom.geom_type == "Polygon":
        return 1
    elif geom.geom_type == "MultiPolygon":
        return len(list(geom.geoms))
    return 0  # fallback if geometrycollection, etc.

current_patch_count = count_patches_in_union(forest_union)
print("Current forest patch count:", current_patch_count)


In [ ]:
# 4. For each afforestable polygon, measure how it changes connectivity
improvements = []
for idx, row in afforestable_areas.iterrows():
    # "Scenario" union: existing forest + this afforestable polygon
    scenario_union = forest_union.union(row.geometry)
    
    scenario_patch_count = count_patches_in_union(scenario_union)
    patch_count_improvement = current_patch_count - scenario_patch_count
    
    improvements.append({
        "idx": idx,
        "patch_count_after": scenario_patch_count,
        "patch_count_improvement": patch_count_improvement
    })

results_df = pd.DataFrame(improvements)
results_df.sort_values("patch_count_improvement", ascending=False, inplace=True)
print(results_df.head(10))


In [ ]:
# 5. Identify top candidates for reforestation (where improvement > 0)
top_candidates = results_df[results_df["patch_count_improvement"] > 0]
candidate_ids = top_candidates["idx"].unique()

print("Afforestable polygons that actually merge or reduce patch counts:")
print(candidate_ids)



In [ ]:
# 6. Plot to see where these top candidates are
fig, ax = plt.subplots(figsize=(10, 8))


In [ ]:
# Existing forest in green
primary_broadleaved_forest_patches.plot(ax=ax, color="green", edgecolor="black", alpha=0.5, label="Existing Forest")

# All afforestable polygons in light grey
afforestable_areas.plot(ax=ax, color="lightgrey", edgecolor="black", alpha=0.5, label="Afforestable Land")

# Highlight top candidates in red
afforestable_areas.loc[candidate_ids].plot(ax=ax, color="red", edgecolor="black", alpha=0.8, label="High Connectivity Potential")

ax.set_title("Afforestable Polygons That Improve Forest Connectivity")
ax.legend()
plt.show()

In [ ]:
# 2. Calculate the distance to the nearest patch
distances = []
for i, row in primary_broadleaved_forest_patches.iterrows():
    # Distance to all OTHER patches
    other_patches = primary_broadleaved_forest_patches.drop(i)
    d_series = other_patches.geometry.distance(row.geometry)
    
    # The nearest distance is the minimum of these distances
    distances.append(d_series.min())

# 3. Store in a new column
primary_broadleaved_forest_patches["nearest_forest_dist_m"] = distances

# 4. Examine results
print(primary_broadleaved_forest_patches[["nearest_forest_dist_m"]].head())

In [ ]:
# Suppose 'primary_broadleaved_forest_patches' has the column "nearest_forest_dist_m"
# which you already calculated.

# 1. Create a figure and axes
fig, ax = plt.subplots(figsize=(10, 8))

# 2. Plot the GeoDataFrame, coloring by the nearest distance
#    - 'column' specifies which column to color by
#    - 'legend=True' adds a colorbar on the side
#    - 'cmap' is the color palette (e.g., "viridis", "plasma", "Reds", etc.)
primary_broadleaved_forest_patches.plot(
    column="nearest_forest_dist_m",
    ax=ax,
    cmap="viridis",
    legend=True,
    edgecolor="black",
    alpha=0.8
)

# 3. (Optional) Label the patches with their nearest distance
#    Using each patch centroid for label placement
for idx, row in primary_broadleaved_forest_patches.iterrows():
    # Get centroid coordinates
    x = row.geometry.centroid.x
    y = row.geometry.centroid.y
    
    # Get the distance value
    dist_val = row["nearest_forest_dist_m"]
    # Ensure it's not NaN (if any patches didn't have a distance)
    if not pd.isna(dist_val):
        # Format the distance (e.g., no decimals or fewer decimals)
        plt.text(x, y, f"{dist_val:.0f}", ha="center", va="center", fontsize=8)

# 4. Add a title (optional)
ax.set_title("Nearest Distance Between Forest Patches (m)", fontsize=14)

plt.show()

In [ ]:
import numpy as np

# We'll store two lists: one for nearest patch ID and one for distance
closest_patch_ids = []
closest_dists = []

for i, patch_i in primary_broadleaved_forest_patches.iterrows():
    # Drop the current patch from the comparison
    other_patches = primary_broadleaved_forest_patches.drop(i)
    
    # Calculate distances from the current patch to all others
    dist_series = other_patches.geometry.distance(patch_i.geometry)
    
    # Identify the minimum distance and the index (ID) of that patch
    min_dist = dist_series.min()
    min_idx = dist_series.idxmin()  # The row index in `other_patches` with the smallest distance
    
    # Append results
    closest_dists.append(min_dist)
    closest_patch_ids.append(min_idx)

# Store the results in new columns
primary_broadleaved_forest_patches["nearest_patch_id"] = closest_patch_ids
primary_broadleaved_forest_patches["nearest_dist_m"] = closest_dists

# Inspect results
print(primary_broadleaved_forest_patches[["nearest_patch_id", "nearest_dist_m"]].head())

In [ ]:
closest_patch_names = []
for i, patch_i in primary_broadleaved_forest_patches.iterrows():
    other_patches = primary_broadleaved_forest_patches.drop(i)
    dist_series = other_patches.geometry.distance(patch_i.geometry)
    
    min_idx = dist_series.idxmin()
    min_dist = dist_series[min_idx]
    
    # Suppose "ID" is the column you want:
    nearest_name = other_patches.loc[min_idx, "OBJECTID"]
    
    closest_patch_names.append(nearest_name)
    # ... store the distance as before ...

In [ ]:
primary_broadleaved_forest_patches["geometry"] = (
    primary_broadleaved_forest_patches.geometry.buffer(0)
)

In [ ]:
primary_broadleaved_forest_patches["geometry"] = (
    primary_broadleaved_forest_patches.geometry.simplify(tolerance=10)
)

In [ ]:
# Disturbed broadleaved forest

In [ ]:
disturbed_broadleaved_forest_patches = terrestrial_landcover[
    terrestrial_landcover["Classify"] == "Disturbed broadleaved forest (Secondary Forest)"
].copy()

In [ ]:
# Calculate the area of each patch
disturbed_broadleaved_forest_patches["area_m2"] = disturbed_broadleaved_forest_patches.geometry.area
# Rank patches by area (largest = rank 1)
disturbed_broadleaved_forest_patches["area_rank"] = disturbed_broadleaved_forest_patches["area_m2"].rank(method="dense", ascending=False)

ranked_disturbed_patches = disturbed_broadleaved_forest_patches.sort_values(by="area_rank")
# Define the columns you want to display
columns_to_display = ["OBJECTID", "Classify", "area_m2", "area_rank", "geometry"]
# Display the ranked patches with the selected columns
display(ranked_disturbed_patches[columns_to_display].head())

In [ ]:
# Discrete colors for the top 5 ranks
top_5_colors = {
    1: "red",   # Largest
    2: "orange",
    3: "yellow",
    4: "green",
    5: "blue"
}

# Fallback colormap for ranks > 5
fallback_cmap = cm.get_cmap("viridis", len(ranked_disturbed_patches) - 5)
fallback_norm = mcolors.Normalize(vmin=6, vmax=ranked_disturbed_patches["area_rank"].max())

# Assign colors: Top 5 get discrete colors, others use fallback colormap, and NaN get grey
def assign_color(rank):
    if np.isnan(rank):  # Check for NaN
        return "grey"   # Color for unranked patches
    elif rank in top_5_colors:
        return top_5_colors[rank]
    else:
        return fallback_cmap(fallback_norm(rank))

# Assign colors to disturbed patches
ranked_disturbed_patches["color"] = ranked_disturbed_patches["area_rank"].apply(assign_color)

# Plot the disturbed patches
fig, ax = plt.subplots(figsize=(10, 8))
base = jamaica_boundary.plot(ax=ax, color="lightgrey", edgecolor="black")  # Plot boundary
ranked_disturbed_patches.plot(ax=ax, color=ranked_disturbed_patches["color"], edgecolor="black", alpha=0.8)

# Add a colorbar for fallback colormap
sm = cm.ScalarMappable(cmap=fallback_cmap, norm=fallback_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.04)
cbar.set_label('Forest Rank > 5', fontsize=12)

# Add a legend for the top 5 ranks and unranked patches
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Rank 1 (Largest)', markerfacecolor='red', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Rank 2', markerfacecolor='orange', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Rank 3', markerfacecolor='yellow', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Rank 4', markerfacecolor='green', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Rank 5', markerfacecolor='blue', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Unranked', markerfacecolor='grey', markersize=10)  # For unranked
]
ax.legend(handles=legend_elements, title="Forest Size Rank", loc="upper left")

# Add title and labels
ax.set_title("Ranked Disturbed Broadleaved Forest Patches", fontsize=16)
ax.set_xlabel("Easting (meters)")
ax.set_ylabel("Northing (meters)")

plt.show()

# ---- Option A: Distance to the nearest patch ----

In [ ]:

#
# 6a. Calculate distance to the nearest primary broadleaved forest patch (excluding itself)
#     For each patch, find the distance to all others, then take the min.


In [ ]:
print(primary_broadleaved_forest_patches.crs)

In [ ]:
distances = []
for i, patch_i in primary_broadleaved_forest_patches.iterrows():
    # Distance from patch_i to all patches in the GeoDataFrame (including itself)
    dists_i = primary_broadleaved_forest_patches.geometry.distance(patch_i.geometry)
    # Exclude itself by setting distance to itself as NaN (or remove it from the series)
    dists_i[i] = float("inf")
    # Get the minimum non-zero distance
    distances.append(dists_i.min())

primary_broadleaved_forest_patches["nearest_forest_dist_m"] = distances

In [ ]:
print(primary_broadleaved_forest_patches[["area_rank", "nearest_forest_dist_m"]].head())

In [ ]:

# Ensure CRS is correct
print("CRS:", primary_broadleaved_forest_patches.crs)

# ---- Step 1: Debugging Geometries ----

# Check for invalid geometries
print("Valid geometries before fixing:", primary_broadleaved_forest_patches.is_valid.sum(), "/", len(primary_broadleaved_forest_patches))

# Fix invalid geometries (buffer(0))
primary_broadleaved_forest_patches["geometry"] = primary_broadleaved_forest_patches.geometry.buffer(0)

# Verify all geometries are valid
print("Valid geometries after fixing:", primary_broadleaved_forest_patches.is_valid.sum(), "/", len(primary_broadleaved_forest_patches))

# Inspect geometry bounds (extents) to ensure no anomalies
print("Bounds of geometries:", primary_broadleaved_forest_patches.total_bounds)

# Check areas of geometries to identify extreme outliers
print("Area stats:")
print(primary_broadleaved_forest_patches.geometry.area.describe())

# ---- Step 2: Simplify Geometries Further ----

# Simplify complex geometries to reduce calculation errors
primary_broadleaved_forest_patches["geometry"] = primary_broadleaved_forest_patches.geometry.simplify(tolerance=20)

# Ensure no empty geometries
primary_broadleaved_forest_patches = primary_broadleaved_forest_patches[~primary_broadleaved_forest_patches.geometry.is_empty]

# ---- Step 3: Spatial Index Optimization for Distance Calculation ----

# Create spatial index
spatial_index = primary_broadleaved_forest_patches.sindex

# Define function to calculate nearest distance using bounding boxes
def calculate_nearest_distance_bbox(geometry, sindex, gdf):
    """
    Calculate the distance to the nearest geometry using bounding boxes.

    Parameters:
    - geometry: Shapely geometry (the current geometry to compare).
    - sindex: Spatial index of the GeoDataFrame.
    - gdf: GeoDataFrame containing the geometries.

    Returns:
    - Distance to the nearest bounding box, excluding self.
    """
    try:
        # Find the index of the nearest geometry using the spatial index
        nearest_index = list(sindex.nearest([geometry.bounds], return_all=False))[0]
        
        # Retrieve possible matches
        possible_matches = gdf.iloc[nearest_index]
        
        # Exclude the current geometry itself
        possible_matches = possible_matches[possible_matches.geometry != geometry]
        
        if not possible_matches.empty:
            # Use bounding box distance as an approximation
            bbox_distance = geometry.bounds.distance(possible_matches.geometry.unary_union.bounds)
            return bbox_distance
        else:
            return np.nan
    except Exception as e:
        print(f"Error in bounding box distance calculation for {geometry}: {e}")
        return np.nan

# ---- Step 4: Apply Bounding Box Distance Calculation ----

primary_broadleaved_forest_patches["nearest_forest_dist_bbox"] = primary_broadleaved_forest_patches.geometry.apply(
    lambda geom: calculate_nearest_distance_bbox(geom, spatial_index, primary_broadleaved_forest_patches)
    if geom.is_valid and not geom.is_empty else np.nan
)

# ---- Step 5: Debugging Results ----

# Check descriptive statistics of the calculated distances
print("Bounding Box Distance Stats:")
print(primary_broadleaved_forest_patches["nearest_forest_dist_bbox"].describe())

# Verify that the nearest distances make sense
print(primary_broadleaved_forest_patches[["area_rank", "nearest_forest_dist_bbox"]].head())

In [ ]:
# Descriptive statistics of the distances
print(primary_broadleaved_forest_patches["nearest_forest_dist_bbox"].describe())

# Preview the calculated distances
print(primary_broadleaved_forest_patches[["area_rank", "nearest_forest_dist_bbox"]].head())